# ReachX Trajectory-Speed Figures

This notebook creates trajectory-speed figures from manually curated ReachX reaches. Run the cells from top to bottom.

The notebook reads ReachX output files, but it does not edit ReachX source code, raw data, or ReachX output files.


## 1. Import packages and define constants

These constants describe ReachX result codes and the trajectory columns used for fixed-cam and legacy-cam outputs.


In [ ]:
from pathlib import Path
import re

from matplotlib import colors as mcolors
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
import numpy as np

try:
    import plotly.graph_objects as go
except ImportError:
    go = None

RESULT_NAMES = {
    2: "grabbed",
    3: "missed",
    4: "dropped",
    5: "stalled",
}

HAND_POS_NAMES = {
    0: "unspecified",
    1: "left",
    2: "right",
    3: "above",
    4: "below",
}

FIXED_TRAJ_COLUMNS = {
    "x": 0,
    "y": 1,
    "z": 2,
    "speed": 6,
}

LEGACY_TRAJ_COLUMNS = {
    "x": 6,
    "y": 1,
    "z": 3,
    "speed": 10,
}

POINTS_PER_FRAME_PAIR = 8


## 2. Set the project folder and session folder

Set `SESSION_FOLDER` to the ReachX session folder you want to analyze. This should be the folder that directly contains the curated `*_reaches.txt` file.


In [ ]:
# This lets the notebook work when VS Code starts in either the repo root or the velo-example folder.
project_folder = Path.cwd()
if project_folder.name != "velo-example" and (project_folder / "velo-example").is_dir():
    project_folder = project_folder / "velo-example"

# Paste your ReachX session folder path here.
# Example: SESSION_FOLDER = r"C:\Users\your-name\Documents\reachx\session001"
SESSION_FOLDER = None

# Use "auto" unless the selected session contains both fixed-cam and legacy-cam outputs.
WORKSPACE_CHOICE = "auto"  # Options: "auto", "fixed", "legacy"

print(f"Project folder: {project_folder}")
print(f"Session folder setting: {SESSION_FOLDER}")


## 3. Detect the ReachX workspace format

Fixed-cam sessions have a scorer folder containing `trajectories.npz`. Legacy-cam sessions have a scorer folder containing `hand.npy`.


In [ ]:
if SESSION_FOLDER is None or str(SESSION_FOLDER).strip() == "":
    raise ValueError("Set SESSION_FOLDER in step 2 before continuing.")

session_folder = Path(SESSION_FOLDER).expanduser()
if not session_folder.is_dir():
    raise FileNotFoundError(f"Session folder does not exist: {session_folder}")

fixed_scorer_folders = sorted(
    folder
    for folder in session_folder.iterdir()
    if folder.is_dir() and (folder / "trajectories.npz").is_file()
)

legacy_scorer_folders = sorted(
    folder
    for folder in session_folder.iterdir()
    if folder.is_dir() and (folder / "hand.npy").is_file()
)

detected_formats = []
if fixed_scorer_folders:
    detected_formats.append("fixed")
if legacy_scorer_folders:
    detected_formats.append("legacy")

workspace_aliases = {
    "": "auto",
    "a": "auto",
    "auto": "auto",
    "f": "fixed",
    "fixed": "fixed",
    "fixed-cam": "fixed",
    "l": "legacy",
    "legacy": "legacy",
    "legacy-cam": "legacy",
}

workspace_choice = str(WORKSPACE_CHOICE).strip().lower()
if workspace_choice not in workspace_aliases:
    raise ValueError(f"Unknown workspace option: {WORKSPACE_CHOICE}")

requested_workspace = workspace_aliases[workspace_choice]

if requested_workspace == "auto":
    if len(detected_formats) == 1:
        workspace_format = detected_formats[0]
    elif len(detected_formats) == 0:
        raise ValueError(
            "Could not detect fixed-cam or legacy ReachX trajectory output. "
            "Expected one scorer folder with trajectories.npz for fixed-cam "
            "or hand.npy for legacy-cam."
        )
    else:
        raise ValueError(
            "Both fixed-cam and legacy trajectory outputs were detected. "
            "Set WORKSPACE_CHOICE to 'fixed' or 'legacy'."
        )
else:
    if requested_workspace not in detected_formats:
        detected_text = ", ".join(detected_formats) if detected_formats else "none"
        raise ValueError(
            f"Manual selection requested {requested_workspace}, but detected formats are: {detected_text}."
        )
    workspace_format = requested_workspace

print(f"Detected formats: {detected_formats}")
print(f"Using workspace format: {workspace_format}")


## 4. Find the curated reach file

The notebook uses exactly one session-level `*_reaches.txt` file. It rejects `detected_reaches.txt` so algorithm-scored reaches are not used by accident.


In [ ]:
reach_file_candidates = []

for path in session_folder.glob("*_reaches.txt"):
    if path.name == "detected_reaches.txt":
        continue
    if "detected" in path.name.lower():
        continue
    if path.is_file():
        reach_file_candidates.append(path)

if len(reach_file_candidates) == 0:
    raise FileNotFoundError(
        "No session-level curated reach file was found. "
        "Expected exactly one *_reaches.txt file directly inside the session folder."
    )

if len(reach_file_candidates) > 1:
    candidate_names = ", ".join(path.name for path in reach_file_candidates)
    raise ValueError(f"Multiple possible curated reach files were found: {candidate_names}")

reach_file = reach_file_candidates[0]
print(f"Curated reach file: {reach_file}")


## 5. Load and validate curated reaches

Each reach row should contain four or five integer columns. Four-column rows are treated as older files with an unspecified hand position.


In [ ]:
reaches = []

with reach_file.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        parts = stripped.split()
        if len(parts) not in (4, 5):
            raise ValueError(
                f"{reach_file.name} line {line_number} has {len(parts)} columns; "
                "expected 4 or 5 integer columns."
            )

        try:
            values = [int(part) for part in parts]
        except ValueError as exc:
            raise ValueError(f"{reach_file.name} line {line_number} contains a non-integer value.") from exc

        if len(values) == 4:
            values.append(0)

        frame, max_delta, duration, result, hand_pos = values

        if frame < 0:
            raise ValueError(f"{reach_file.name} line {line_number} has a negative frame number.")
        if max_delta < 0:
            raise ValueError(f"{reach_file.name} line {line_number} has a negative max_delta.")
        if duration <= 0:
            raise ValueError(f"{reach_file.name} line {line_number} has a non-positive duration.")
        if max_delta > duration:
            raise ValueError(f"{reach_file.name} line {line_number} has max_delta greater than duration.")
        if result not in RESULT_NAMES:
            raise ValueError(
                f"{reach_file.name} line {line_number} has result {result}; "
                "expected a reach result code from 2 through 5."
            )
        if hand_pos not in HAND_POS_NAMES:
            raise ValueError(
                f"{reach_file.name} line {line_number} has hand_pos {hand_pos}; "
                "expected 0 through 4."
            )

        reaches.append({
            "frame": frame,
            "max_delta": max_delta,
            "duration": duration,
            "result": result,
            "hand_pos": hand_pos,
            "line_number": line_number,
        })

if not reaches:
    raise ValueError(f"Curated reach file contains no reach rows: {reach_file}")

reaches = sorted(reaches, key=lambda reach: reach["frame"])

previous_reach = None
for reach in reaches:
    reach_end_frame = reach["frame"] + reach["duration"]
    if previous_reach is not None:
        previous_end_frame = previous_reach["frame"] + previous_reach["duration"]
        if reach["frame"] <= previous_end_frame:
            raise ValueError(
                f"Curated reaches overlap in {reach_file.name}: "
                f"line {previous_reach['line_number']} and line {reach['line_number']}."
            )
    previous_reach = reach

for index, reach in enumerate(reaches, start=1):
    reach["index"] = index

print(f"Loaded {len(reaches)} curated reaches.")


## 6. Load the right-hand trajectory

The notebook loads the right-hand trajectory from the one scorer folder that matches the selected workspace format.


In [ ]:
if workspace_format == "fixed":
    scorer_folders = fixed_scorer_folders
else:
    scorer_folders = legacy_scorer_folders

if len(scorer_folders) == 0:
    raise FileNotFoundError(f"No {workspace_format} scorer folder with trajectory output was found.")

if len(scorer_folders) > 1:
    scorer_names = ", ".join(path.name for path in scorer_folders)
    raise ValueError(
        f"Multiple {workspace_format} scorer folders with trajectory output were found: {scorer_names}. "
        "Choose a session folder with one clear trajectory output or remove ambiguity before running."
    )

scorer_folder = scorer_folders[0]

if workspace_format == "fixed":
    trajectory_file = scorer_folder / "trajectories.npz"
    with np.load(trajectory_file) as data:
        if "R_Hand" not in data:
            raise ValueError(f"{trajectory_file.name} does not contain the fixed-cam R_Hand trajectory.")
        raw_trajectory = data["R_Hand"]

    if raw_trajectory.ndim != 2:
        raise ValueError(f"{trajectory_file.name} is not a 2D trajectory array.")
    if raw_trajectory.shape[1] < 7:
        raise ValueError(f"{trajectory_file.name} has {raw_trajectory.shape[1]} columns; expected at least 7.")

    trajectory = raw_trajectory[:, [
        FIXED_TRAJ_COLUMNS["x"],
        FIXED_TRAJ_COLUMNS["y"],
        FIXED_TRAJ_COLUMNS["z"],
        FIXED_TRAJ_COLUMNS["speed"],
    ]]
    speed_units = "mm/ms"

else:
    trajectory_file = scorer_folder / "hand.npy"
    raw_trajectory = np.load(trajectory_file)

    if raw_trajectory.ndim != 2:
        raise ValueError(f"{trajectory_file.name} is not a 2D trajectory array.")
    if raw_trajectory.shape[1] < 11:
        raise ValueError(f"{trajectory_file.name} has {raw_trajectory.shape[1]} columns; expected at least 11.")

    trajectory = raw_trajectory[:, [
        LEGACY_TRAJ_COLUMNS["x"],
        LEGACY_TRAJ_COLUMNS["y"],
        LEGACY_TRAJ_COLUMNS["z"],
        LEGACY_TRAJ_COLUMNS["speed"],
    ]]
    speed_units = "mm/s"

if trajectory.shape[0] < 2:
    raise ValueError(f"{trajectory_file.name} has fewer than two trajectory frames.")
if not np.isfinite(trajectory).all():
    raise ValueError(f"{trajectory_file.name} contains non-finite trajectory values.")

print(f"Scorer folder: {scorer_folder}")
print(f"Trajectory file: {trajectory_file}")
print(f"Trajectory shape: {trajectory.shape}")
print(f"Speed units: {speed_units}")


## 7. Review the curated reaches

Use this table to choose a reach by its displayed index or by its start frame.


In [ ]:
print("Curated reaches:")
for reach in reaches:
    reach_end_frame = reach["frame"] + reach["duration"]
    result_name = RESULT_NAMES[reach["result"]]
    hand_position = HAND_POS_NAMES[reach["hand_pos"]]
    print(
        f"  {reach['index']:>3}. "
        f"frame={reach['frame']} "
        f"end={reach_end_frame} "
        f"result={result_name} "
        f"hand_pos={hand_position}"
    )


## 8. Choose what to plot

For a single reach, `SELECTED_REACH` can be either the displayed reach index or the reach start frame.


In [ ]:
PLOT_SCOPE = "one"      # Options: "one", "all"
SELECTED_REACH = 20     # Used only when PLOT_SCOPE is "one". This can be a reach index or start frame.
PLOT_VIEW = "2d"        # Options: "2d", "3d"

plot_scope = str(PLOT_SCOPE).strip().lower()
plot_view = str(PLOT_VIEW).strip().lower()

if plot_view in ("2", "2d"):
    plot_view = "2d"
elif plot_view in ("3", "3d"):
    plot_view = "3d"
else:
    raise ValueError(f"Unknown plot view: {PLOT_VIEW}")

if plot_scope in ("one", "1", "single"):
    selected_value = int(SELECTED_REACH)

    if 1 <= selected_value <= len(reaches):
        selected_reaches = [reaches[selected_value - 1]]
    else:
        matching_reaches = [reach for reach in reaches if reach["frame"] == selected_value]
        if len(matching_reaches) == 1:
            selected_reaches = matching_reaches
        elif len(matching_reaches) > 1:
            raise ValueError(f"More than one reach starts at frame {selected_value}.")
        else:
            raise ValueError(f"No curated reach matched index or start frame: {selected_value}")

    plot_scope = "single"

elif plot_scope in ("all", "a"):
    selected_reaches = reaches
    plot_scope = "all"

else:
    raise ValueError(f"Unknown plot scope: {PLOT_SCOPE}")

for reach in selected_reaches:
    reach_end_frame = reach["frame"] + reach["duration"]
    if reach_end_frame >= trajectory.shape[0]:
        raise ValueError(
            f"Curated reach starting at frame {reach['frame']} ends at frame {reach_end_frame}, "
            f"but the trajectory has only {trajectory.shape[0]} frames."
        )

    excerpt = trajectory[reach["frame"]:reach_end_frame + 1, :]
    if excerpt.shape[0] < 2:
        raise ValueError(f"Curated reach starting at frame {reach['frame']} has fewer than two trajectory samples.")
    if not np.isfinite(excerpt).all():
        raise ValueError(f"Curated reach starting at frame {reach['frame']} contains non-finite trajectory values.")

print(f"Plot scope: {plot_scope}")
print(f"Plot view: {plot_view}")
print(f"Selected reaches: {[reach['index'] for reach in selected_reaches]}")


## 9. Prepare figure titles and output paths

Figures are saved in the `figures` folder inside this example project.


In [ ]:
output_dir = project_folder / "figures"
output_dir.mkdir(exist_ok=True)

if reach_file.name.endswith("_reaches.txt"):
    session_label = reach_file.name[:-len("_reaches.txt")]
else:
    session_label = session_folder.name

safe_session = re.sub(r"[^A-Za-z0-9_.-]+", "_", session_label).strip("_")
if not safe_session:
    safe_session = "reachx_session"

if len(selected_reaches) == 1:
    title_reach = (
        f"reach {selected_reaches[0]['index']} "
        f"frame {selected_reaches[0]['frame']} "
        f"({RESULT_NAMES[selected_reaches[0]['result']]})"
    )
else:
    title_reach = f"all curated reaches (n={len(selected_reaches)})"

figure_title = f"{session_label} | {workspace_format} | {plot_view.upper()} | {title_reach}"

if plot_scope == "all":
    output_stem = f"{safe_session}_{workspace_format}_all_curated_reaches_{plot_view}"
else:
    reach = selected_reaches[0]
    if plot_view == "3d":
        output_stem = f"{safe_session}_{workspace_format}_reach{reach['index']:03d}_frame{reach['frame']}_3d_interactive"
    else:
        output_stem = f"{safe_session}_{workspace_format}_reach{reach['index']:03d}_frame{reach['frame']}_2d"

print(f"Figure title: {figure_title}")
print(f"Output folder: {output_dir}")
print(f"Output stem: {output_stem}")


## 10. Plot and save the figure

The path is color-coded by speed. The start point is a circle and the end point is a square.


In [ ]:
speed_arrays = []
for reach in selected_reaches:
    start_frame = reach["frame"]
    end_frame = reach["frame"] + reach["duration"]
    speed_arrays.append(trajectory[start_frame:end_frame + 1, 3])

all_speeds = np.concatenate(speed_arrays)
speed_min = float(np.min(all_speeds))
speed_max = float(np.max(all_speeds))
if speed_min == speed_max:
    speed_max = speed_min + 1.0

output_files = []

if plot_view == "2d":
    speed_norm = mcolors.Normalize(vmin=speed_min, vmax=speed_max)
    fig, axis = plt.subplots(figsize=(7, 6))
    color_source = None

    for reach_index, reach in enumerate(selected_reaches):
        start_frame = reach["frame"]
        end_frame = reach["frame"] + reach["duration"]
        excerpt = trajectory[start_frame:end_frame + 1, :]

        y_values = excerpt[:, 1]
        z_values = excerpt[:, 2]
        speed_values = excerpt[:, 3]
        points = np.column_stack([y_values, z_values])

        smooth_points = []
        smooth_speeds = []
        for index in range(points.shape[0] - 1):
            for step in range(POINTS_PER_FRAME_PAIR):
                fraction = step / POINTS_PER_FRAME_PAIR
                smooth_points.append(points[index] + fraction * (points[index + 1] - points[index]))
                smooth_speeds.append(speed_values[index] + fraction * (speed_values[index + 1] - speed_values[index]))

        smooth_points.append(points[-1])
        smooth_speeds.append(speed_values[-1])
        smooth_points = np.asarray(smooth_points)
        smooth_speeds = np.asarray(smooth_speeds)

        segments = np.stack([smooth_points[:-1], smooth_points[1:]], axis=1)
        segment_speeds = (smooth_speeds[:-1] + smooth_speeds[1:]) / 2.0

        collection = LineCollection(segments, cmap="viridis", norm=speed_norm, linewidth=2.0)
        collection.set_array(segment_speeds)
        axis.add_collection(collection)
        color_source = collection

        start_label = "start" if reach_index == 0 else None
        end_label = "end" if reach_index == 0 else None
        axis.scatter(y_values[0], z_values[0], color="black", s=24, marker="o", label=start_label, zorder=3)
        axis.scatter(y_values[-1], z_values[-1], color="black", s=24, marker="s", label=end_label, zorder=3)

    axis.autoscale()
    axis.set_aspect("equal", adjustable="datalim")
    axis.set_xlabel("Y position (mm)")
    axis.set_ylabel("Z position (mm)")
    axis.set_title(figure_title)
    axis.legend(loc="best")

    colorbar = fig.colorbar(color_source, ax=axis, shrink=0.8)
    colorbar.set_label(f"Speed ({speed_units})")

    png_file = output_dir / f"{output_stem}.png"
    pdf_file = output_dir / f"{output_stem}.pdf"
    fig.tight_layout()
    fig.savefig(png_file, dpi=300)
    fig.savefig(pdf_file)
    output_files = [png_file, pdf_file]
    plt.show()

else:
    if go is None:
        raise ImportError("Plotly is required for live 3D plots. Install it with: conda install plotly")

    fig = go.Figure()

    for reach_index, reach in enumerate(selected_reaches):
        start_frame = reach["frame"]
        end_frame = reach["frame"] + reach["duration"]
        excerpt = trajectory[start_frame:end_frame + 1, :]

        points = excerpt[:, :3]
        speed_values = excerpt[:, 3]

        smooth_points = []
        smooth_speeds = []
        for index in range(points.shape[0] - 1):
            for step in range(POINTS_PER_FRAME_PAIR):
                fraction = step / POINTS_PER_FRAME_PAIR
                smooth_points.append(points[index] + fraction * (points[index + 1] - points[index]))
                smooth_speeds.append(speed_values[index] + fraction * (speed_values[index + 1] - speed_values[index]))

        smooth_points.append(points[-1])
        smooth_speeds.append(speed_values[-1])
        smooth_points = np.asarray(smooth_points)
        smooth_speeds = np.asarray(smooth_speeds)

        trace_name = f"reach {reach['index']} frame {reach['frame']} ({RESULT_NAMES[reach['result']]})"
        line_settings = {
            "color": smooth_speeds,
            "colorscale": "Viridis",
            "cmin": speed_min,
            "cmax": speed_max,
            "width": 6,
            "showscale": reach_index == 0,
        }
        if reach_index == 0:
            line_settings["colorbar"] = {"title": f"Speed ({speed_units})"}

        fig.add_trace(
            go.Scatter3d(
                x=smooth_points[:, 0],
                y=smooth_points[:, 1],
                z=smooth_points[:, 2],
                mode="lines",
                name=trace_name,
                line=line_settings,
                customdata=smooth_speeds,
                hovertemplate=(
                    f"{trace_name}<br>"
                    "X: %{x:.3f} mm<br>"
                    "Y: %{y:.3f} mm<br>"
                    "Z: %{z:.3f} mm<br>"
                    "Speed: %{customdata:.5f}<extra></extra>"
                ),
            )
        )

        show_legend = reach_index == 0
        fig.add_trace(
            go.Scatter3d(
                x=[smooth_points[0, 0]],
                y=[smooth_points[0, 1]],
                z=[smooth_points[0, 2]],
                mode="markers",
                name="start",
                showlegend=show_legend,
                marker={"color": "black", "size": 4, "symbol": "circle"},
                hovertemplate=f"{trace_name} start<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter3d(
                x=[smooth_points[-1, 0]],
                y=[smooth_points[-1, 1]],
                z=[smooth_points[-1, 2]],
                mode="markers",
                name="end",
                showlegend=show_legend,
                marker={"color": "black", "size": 4, "symbol": "square"},
                hovertemplate=f"{trace_name} end<extra></extra>",
            )
        )

    fig.update_layout(
        title=figure_title,
        scene={
            "xaxis_title": "X position (mm)",
            "yaxis_title": "Y position (mm)",
            "zaxis_title": "Z position (mm)",
            "aspectmode": "data",
        },
        legend={"itemsizing": "constant"},
        margin={"l": 0, "r": 0, "t": 50, "b": 0},
    )

    if plot_scope == "all":
        html_stem = f"{safe_session}_{workspace_format}_all_curated_reaches_3d_interactive"
    else:
        html_stem = output_stem

    html_file = output_dir / f"{html_stem}.html"
    fig.write_html(
        str(html_file),
        include_plotlyjs="cdn",
        full_html=True,
        config={
            "displaylogo": False,
            "editable": True,
            "responsive": True,
            "scrollZoom": True,
        },
    )
    output_files = [html_file]
    fig.show()

print("Created figure files:")
for output_file in output_files:
    print(f"  {output_file}")
